# DBSCAN — analiza klastrów i wykrywanie szumu

**Cel szkoleniowy:** pokazać klastrowanie gęstościowe, które:
- nie wymaga podawania liczby klastrów,
- potrafi wykrywać klastry o nieregularnych kształtach,
- potrafi oznaczać obserwacje odstające jako **szum**.

W notebooku:
1. wygenerujemy dane o nieregularnych kształtach,
2. przeskalujemy cechy,
3. zastosujemy DBSCAN,
4. pokażemy znaczenie `eps` i `min_samples`,
5. przeanalizujemy szum,
6. porównamy wyniki dla różnych parametrów.

## 1. Intuicja działania DBSCAN

DBSCAN grupuje punkty na podstawie **gęstości**.

Każdy punkt może być:
- **core point** — ma wystarczająco wielu sąsiadów,
- **border point** — leży blisko punktów rdzeniowych,
- **noise** — nie należy do żadnego klastra.

Dwa najważniejsze parametry:

### `eps`
Maksymalna odległość, w której szukamy sąsiadów.

### `min_samples`
Minimalna liczba punktów potrzebna do utworzenia lokalnego obszaru o odpowiedniej gęstości.

W DBSCAN etykieta `-1` oznacza **szum / outlier**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42

## 2. Dane testowe — dwa półksiężyce

To bardzo dobry przykład dydaktyczny, bo klastry nie są kuliste.

K-Means może mieć z takim układem problem, natomiast DBSCAN zwykle radzi sobie bardzo dobrze.

In [ ]:
X, _ = make_moons(
    n_samples=500,
    noise=0.08,
    random_state=RANDOM_STATE
)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.7)
plt.title("Dane wejściowe — make_moons")
plt.xlabel("Cecha 1")
plt.ylabel("Cecha 2")
plt.show()

## 3. Skalowanie danych

DBSCAN bazuje na odległościach, więc skala cech ma duże znaczenie.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=["x1", "x2"]).describe().round(2)

## 4. Pierwszy model DBSCAN

Na początek:
- `eps = 0.25`
- `min_samples = 5`

In [ ]:
dbscan = DBSCAN(
    eps=0.25,
    min_samples=5
)

labels = dbscan.fit_predict(X_scaled)

unique_labels = np.unique(labels)
print("Etykiety:", unique_labels)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = np.sum(labels == -1)

print("Liczba klastrów:", n_clusters)
print("Liczba punktów szumu:", n_noise)

## 5. Wizualizacja klastrów

In [ ]:
plt.figure(figsize=(8, 6))

for label in sorted(set(labels)):
    mask = labels == label

    if label == -1:
        name = "Szum"
    else:
        name = f"Klaster {label}"

    plt.scatter(
        X_scaled[mask, 0],
        X_scaled[mask, 1],
        label=name,
        alpha=0.75
    )

plt.title("DBSCAN — wynik klastrowania")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

## 6. Silhouette score

Silhouette score można policzyć tylko wtedy, gdy algorytm znalazł co najmniej dwa klastry.

Punkty oznaczone jako `-1` możemy pominąć przy obliczaniu jakości struktury klastrów.

In [ ]:
mask = labels != -1
valid_labels = labels[mask]

if len(set(valid_labels)) >= 2:
    score = silhouette_score(X_scaled[mask], valid_labels)
    print("Silhouette score:", round(score, 4))
else:
    print("Za mało klastrów do obliczenia silhouette score.")

## 7. Eksperyment z parametrem `eps`

To najważniejszy fragment szkoleniowy.

Zbyt małe `eps`:
- wiele punktów zostanie uznanych za szum,
- może powstać zbyt wiele małych klastrów.

Zbyt duże `eps`:
- różne klastry mogą się połączyć w jeden.

In [ ]:
eps_values = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

results = []

for eps in eps_values:
    model = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels_eps = model.fit_predict(X_scaled)

    n_clusters = len(set(labels_eps)) - (1 if -1 in labels_eps else 0)
    n_noise = np.sum(labels_eps == -1)

    score = np.nan

    mask = labels_eps != -1

    if len(set(labels_eps[mask])) >= 2:
        score = silhouette_score(
            X_scaled[mask],
            labels_eps[mask]
        )

    results.append({
        "eps": eps,
        "liczba_klastrow": n_clusters,
        "szum": n_noise,
        "silhouette": score
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    results_df["eps"],
    results_df["liczba_klastrow"],
    marker="o"
)
plt.xlabel("eps")
plt.ylabel("Liczba klastrów")
plt.title("Wpływ eps na liczbę klastrów")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    results_df["eps"],
    results_df["szum"],
    marker="o"
)
plt.xlabel("eps")
plt.ylabel("Liczba punktów szumu")
plt.title("Wpływ eps na liczbę outlierów")
plt.show()

## 8. Eksperyment z `min_samples`

Większe `min_samples` oznacza bardziej rygorystyczne wymagania dotyczące gęstości.

In [ ]:
min_samples_values = [3, 5, 8, 10, 15, 20]

results_min_samples = []

for ms in min_samples_values:
    model = DBSCAN(
        eps=0.25,
        min_samples=ms
    )

    labels_ms = model.fit_predict(X_scaled)

    n_clusters = len(set(labels_ms)) - (1 if -1 in labels_ms else 0)
    n_noise = np.sum(labels_ms == -1)

    results_min_samples.append({
        "min_samples": ms,
        "liczba_klastrow": n_clusters,
        "szum": n_noise
    })

pd.DataFrame(results_min_samples)

## 9. Wizualne porównanie kilku wartości `eps`

In [ ]:
for eps in [0.12, 0.20, 0.25, 0.40]:
    model = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels_tmp = model.fit_predict(X_scaled)

    plt.figure(figsize=(7, 5))

    for label in sorted(set(labels_tmp)):
        mask = labels_tmp == label

        name = "Szum" if label == -1 else f"Klaster {label}"

        plt.scatter(
            X_scaled[mask, 0],
            X_scaled[mask, 1],
            label=name,
            alpha=0.75
        )

    plt.title(f"DBSCAN — eps = {eps}")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.show()

## 10. Wykrywanie obserwacji odstających

DBSCAN może służyć nie tylko do segmentacji, ale również do identyfikacji punktów nietypowych.

Obserwacje oznaczone jako `-1` można potraktować jako kandydatów na anomalie.

In [ ]:
noise_points = X_scaled[labels == -1]

print("Liczba wykrytych obserwacji odstających:", len(noise_points))

pd.DataFrame(
    noise_points,
    columns=["x1", "x2"]
).head(10)

## 11. Ćwiczenia dla uczestników

### Ćwiczenie 1
Zmień:

```python
eps=0.15
```

Sprawdź:
- liczbę klastrów,
- liczbę punktów szumu.

### Ćwiczenie 2
Zmień:

```python
eps=0.40
```

Czy klastry zaczynają się łączyć?

### Ćwiczenie 3
Ustaw:

```python
min_samples=15
```

Jak zmienia się liczba punktów odstających?

### Ćwiczenie 4
Spróbuj dobrać takie wartości parametrów, aby:
- algorytm znalazł dokładnie 2 klastry,
- liczba punktów szumu była możliwie mała.

# Podsumowanie

## DBSCAN — zalety

- nie trzeba znać liczby klastrów,
- dobrze wykrywa klastry o nieregularnych kształtach,
- potrafi wykrywać obserwacje odstające,
- nie zakłada kulistego kształtu klastrów.

## DBSCAN — ograniczenia

- bardzo zależy od parametru `eps`,
- ma problemy, gdy klastry mają bardzo różne gęstości,
- w wysokowymiarowych danych odległości mogą być mało informacyjne,
- skalowanie cech jest zazwyczaj konieczne.

## DBSCAN vs K-Means

**K-Means**
- wymaga liczby klastrów `k`,
- dobrze działa dla klastrów zbliżonych do kulistych,
- nie wykrywa szumu.

**DBSCAN**
- sam określa liczbę klastrów,
- radzi sobie z nieregularnymi kształtami,
- wykrywa szum i outliery.